# Download data

In [ ]:
%pip install -U huggingface_hub

!hf download doem1997/cs604-course-data \
  --repo-type dataset \
  --local-dir /content/cs604-course-data

!tar -xzf /content/cs604-course-data/train.tar.gz -C /content/cs604-course-data
!tar -xzf /content/cs604-course-data/val.tar.gz -C /content/cs604-course-data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 28.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.0.0 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.0.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.0.0 which 

# EDA

## View several images from each of a few classes


In [ ]:
''' randomly chooses 5 classes, then displays 3 training images from each class'''

from pathlib import Path
from collections import defaultdict
import random
import matplotlib.pyplot as plt
from PIL import Image

# locate the label file
root = Path("/content/cs604-course-data")
label_file = root / "train.txt"

# group image paths by class
paths_by_class = defaultdict(list)
with label_file.open() as f:
    for line in f:
        path, class_id = line.strip().split()
        paths_by_class[int(class_id)].append(path)

# uses a fixed random seed
# randomly choose classes
rng = random.Random(42)
selected_classes = rng.sample(sorted(paths_by_class), 5)

fig, axes = plt.subplots(5, 3, figsize=(10, 15))

# for each chosen class, select 3 different iamge paths
for row, class_id in enumerate(selected_classes):
    selected_paths = rng.sample(paths_by_class[class_id], 3)
    for col, path in enumerate(selected_paths):
        with Image.open(root / path) as image:
            axes[row, col].imshow(image.convert("RGB"))
            axes[row, col].set_title(
                f"Class {class_id} | {image.width}×{image.height}"
            )
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## Check image properties on a sample

In [ ]:
''' check 1000 randomly selected training images fro their dimensions, color format, and whether PIL can read them'''
from collections import Counter

all_paths = [path for paths in paths_by_class.values() for path in paths]
sample_paths = random.Random(123).sample(all_paths, 1000)

sizes = Counter()
modes = Counter()
unreadable = []

for path in sample_paths:
    try:
        with Image.open(root / path) as image:
            sizes[image.size] += 1
            modes[image.mode] += 1
            image.verify()
    except Exception as exc:
        unreadable.append((path, str(exc)))

print("Most common sizes:", sizes.most_common(10)) # ((width, height), count)
print("Color modes:", modes)
print("Unreadable images in sample:", len(unreadable))
print("Examples of errors:", unreadable[:5])

Most common sizes: [((375, 500), 283), ((500, 375), 237), ((500, 333), 100), ((281, 500), 43), ((500, 500), 30), ((500, 281), 24), ((333, 500), 18), ((500, 334), 18), ((500, 332), 9), ((500, 331), 8)]
Color modes: Counter({'RGB': 1000})
Unreadable images in sample: 0
Examples of errors: []


# Full fine-tuning baseline

In [ ]:
# Install packages and check paths
%pip install -U timm

from pathlib import Path

ROOT = Path("/content/cs604-course-data")
assert (ROOT / "train.txt").is_file()
assert (ROOT / "val.txt").is_file()
assert (ROOT / "train").is_dir()
assert (ROOT / "val").is_dir()

print("Dataset found at:", ROOT)

Dataset found at: /content/cs604-course-data


In [ ]:
# build the dataset and model
import random
import time
import torch
import timm

from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda"
assert torch.cuda.is_available(), "Select a GPU runtime first"

MODEL_NAME = "hf_hub:timm/vit_base_patch16_224.augreg2_in21k_ft_in1k"

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=10_000,
).to(DEVICE)

# Full fine-tuning: every parameter, including the new head, is trainable.
assert all(p.requires_grad for p in model.parameters())

data_config = timm.data.resolve_model_data_config(model)

# prepares training images at the model's expected size and applies random augmentation
train_transform = timm.data.create_transform(
    **data_config, is_training=True
)

# prepares validation images consistently, without random training augmentation
eval_transform = timm.data.create_transform(
    **data_config, is_training=False
)

class TrainDataset(Dataset):
    def __init__(self, root, label_file, transform):
        self.root = root
        self.transform = transform
        self.rows = []

        with label_file.open() as f:
            for line in f:
                path, class_id = line.strip().split()
                self.rows.append((path, int(class_id)))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        path, class_id = self.rows[index]
        with Image.open(self.root / path) as image:
            x = self.transform(image.convert("RGB"))
        return x, class_id

train_dataset = TrainDataset(ROOT, ROOT / "train.txt", train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

print("Training images:", len(train_dataset))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Training images: 450000
Trainable parameters: 93488656


In [ ]:
# train briefly first, then run the first baseline
from google.colab import drive

drive.mount("/content/drive")

SAVE_DIR = Path("/content/drive/MyDrive/CS604")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = SAVE_DIR / "full_finetune_vit_b16.pt"

SMOKE_TEST = False              # Change to False for the real run
MAX_TRAIN_SECONDS = 2 * 60 * 60  # Conservative 2-hour A100 limit

criterion = nn.CrossEntropyLoss()

# Let the new, randomly initialized classifier learn faster than the backbone.
head = model.get_classifier()
head_ids = {id(p) for p in head.parameters()}

optimizer = torch.optim.AdamW(
    [
        {
            "params": [p for p in model.parameters() if id(p) not in head_ids],
            "lr": 1e-5,
        },
        {
            "params": list(head.parameters()),
            "lr": 1e-3,
        },
    ],
    weight_decay=0.05,
)

model.train()
torch.cuda.reset_peak_memory_stats()
start = time.monotonic()
last_save = start
step = 0
loss_log = []

for epoch in range(10):  # The time limit usually stops this earlier.
    for images, labels in train_loader:
        if time.monotonic() - start >= MAX_TRAIN_SECONDS:
            break

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # A100 supports bf16 mixed precision.
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(images)
            loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        step += 1
        loss_log.append((step, float(loss.item())))

        if step % 100 == 0:
            elapsed = time.monotonic() - start
            print(f"Step {step} | loss {loss.item():.4f} | {elapsed/60:.1f} min")

        # Save periodically in case Colab disconnects.
        if not SMOKE_TEST and time.monotonic() - last_save >= 30 * 60:
            torch.save(model.state_dict(), CHECKPOINT)
            last_save = time.monotonic()
            print("Checkpoint saved")

        if SMOKE_TEST and step >= 5:
            break

    if SMOKE_TEST and step >= 5:
        break
    if time.monotonic() - start >= MAX_TRAIN_SECONDS:
        break

elapsed = time.monotonic() - start
peak_gb = torch.cuda.max_memory_allocated() / 1024**3

print(f"Steps: {step}")
print(f"Training time: {elapsed/60:.1f} minutes")
print(f"Peak PyTorch GPU memory: {peak_gb:.2f} GiB")

if not SMOKE_TEST:
    torch.save(model.state_dict(), CHECKPOINT)
    print("Final checkpoint:", CHECKPOINT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 100 | loss 8.9551 | 0.1 min
Step 200 | loss 8.1553 | 0.2 min
Step 300 | loss 9.0381 | 0.3 min
Step 400 | loss 8.1992 | 0.4 min
Step 500 | loss 7.5093 | 0.5 min
Step 600 | loss 6.4510 | 0.5 min
Step 700 | loss 6.2662 | 0.6 min
Step 800 | loss 7.2639 | 0.7 min
Step 900 | loss 5.7402 | 0.8 min
Step 1000 | loss 6.8102 | 0.9 min
Step 1100 | loss 5.3541 | 1.0 min
Step 1200 | loss 5.0697 | 1.1 min
Step 1300 | loss 5.1155 | 1.2 min
Step 1400 | loss 5.0621 | 1.3 min
Step 1500 | loss 4.5010 | 1.4 min
Step 1600 | loss 4.6726 | 1.5 min
Step 1700 | loss 5.4116 | 1.5 min
Step 1800 | loss 4.6325 | 1.6 min
Step 1900 | loss 4.2670 | 1.7 min
Step 2000 | loss 3.8056 | 1.8 min
Step 2100 | loss 4.0983 | 1.9 min
Step 2200 | loss 4.7578 | 2.0 min
Step 2300 | loss 3.4427 | 2.1 min
Step 2400 | loss 3.9483 | 2.2 min
Step 2500 | loss 4.2162 | 2.3 min
Step 2600 | loss 2.9828 | 2.4 

In [ ]:
# generate validation predictions
from zipfile import ZipFile, ZIP_DEFLATED

class ValDataset(Dataset):
    def __init__(self, root, path_file, transform):
        self.root = root
        self.transform = transform
        self.paths = path_file.read_text().splitlines()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        with Image.open(self.root / path) as image:
            x = self.transform(image.convert("RGB"))
        return x, path

val_dataset = ValDataset(ROOT, ROOT / "val.txt", eval_transform)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

model.eval()
predictions = []

with torch.inference_mode():
    for images, paths in val_loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(images)

        class_ids = logits.argmax(dim=1).cpu().tolist()
        predictions.extend(zip(paths, class_ids))

assert len(predictions) == 50_000
assert len({path for path, _ in predictions}) == 50_000
assert all(0 <= class_id < 10_000 for _, class_id in predictions)

TEAM_NUMBER = "08"  # Replace with your actual team number
txt_path = SAVE_DIR / f"val_team_{TEAM_NUMBER}.txt"
zip_path = SAVE_DIR / f"val_team_{TEAM_NUMBER}.zip"

with txt_path.open("w") as f:
    for path, class_id in predictions:
        f.write(f"{path} {class_id}\n")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as zf:
    zf.write(txt_path, arcname=txt_path.name)

print("Submission:", zip_path)
print("Lines:", len(predictions))

Submission: /content/drive/MyDrive/CS604/val_team_08.zip
Lines: 50000
